In [7]:
import pandas as pd
import tiktoken
from tqdm import tqdm








# To get the tokeniser corresponding to a specific model in the OpenAI API:
enc = tiktoken.get_encoding("o200k_base")

enc.encode("tiktoken is great!")

[83, 8251, 2488, 382, 2212, 0]

In [ ]:
def token_counter(text):
    return len(enc.encode(text))


print(token_counter("i love cows"))




3


In [9]:
csv_paths = ["/Users/josh/Documents/symptom-extraction-demo/data/status_data.csv"]

In [11]:
tqdm.pandas()
total_count=0
for _ in csv_paths:
    #df=pd.read_csv(_)
    df=pd.read_csv(_)
    df['tokens'] = df['text'].progress_apply(token_counter)
    count=df['tokens'].sum()
    total_count += count


print ("!!! done !!!!")

100%|██████████| 3085/3085 [00:00<00:00, 7630.06it/s]

!!! done !!!!


In [13]:
print('millions of tokens')
print(total_count/ 1_000_000)
print('cost in $')
print((total_count/1_000_000)*2)

millions of tokens
0.485948
cost in $
0.971896


In [4]:
from openai import OpenAI
from pydantic import BaseModel
import os
from dotenv import load_dotenv
from src.schema import OutputSchema
from src.llm import openai_chat_completion_response


# create client
"""client = OpenAI(
    api_key='ollama',
    max_retries=3,
    # if using openai use base url 'https://api.openai.com/v1'
    base_url='http://localhost:11434/v1'
)"""

# Load .env into environment
load_dotenv()
# Pull API key from environment
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Please set the OPENAI_API_KEY environment variable in your .env file")


client = OpenAI(
    api_key=api_key,
    max_retries=3,
)

In [6]:
print(type(research_paper1))
print(type(research_paper))

<class 'dict'>
<class 'str'>


In [5]:
class ResearchPaperExtraction(BaseModel):
    title: str

  

response = client.responses.parse(
    model="gpt-4.1-nano-2025-04-14",
    input=[
        {
            "role": "system",
            "content": "You are an expert at structured data extraction. the paper is called an abundence of possums"},
        {"role": "user", "content": "..."},
    ],
    text_format=ResearchPaperExtraction,
)

research_paper = response.output_text
research_paper1 = json.loads(research_paper)

In [7]:
print(type(research_paper1))
print(type(research_paper))

<class 'dict'>
<class 'str'>


In [8]:
print(research_paper1)

{'title': 'An Abundance of Possums'}


In [23]:
from openai import OpenAI

client = OpenAI()

tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current temperature for a given location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City and country e.g. Bogotá, Colombia"
                }
            },
            "required": [
                "location"
            ],
            "additionalProperties": False
        },
        "strict": True
    }
}]

completion = client.chat.completions.create(
    model="gpt-4.1-nano-2025-04-14",
    messages=[{"role": "user", "content": "What is the weather like in Paris today?"}],
    tools=tools
)

print(completion.choices[0].message.tool_calls)

[ChatCompletionMessageToolCall(id='call_t3CRiyTsZZPCZ237fedBGsnY', function=Function(arguments='{"location":"Paris"}', name='get_weather'), type='function')]


In [26]:
import json

output = completion.choices[0].message.tool_calls[0].function.arguments
output = json.loads(output)
print(output)

{'location': 'Paris'}


In [18]:
from pydantic import BaseModel
from openai import OpenAI
import os, json

client = OpenAI(api_key=os.environ.get("TOGETHER_API_KEY"),
                base_url="https://api.together.xyz/v1",)



completion = client.chat.completions.create(
    model="meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo",
    messages=[
        {"role": "system", "content": "Extract the event information."},
        {"role": "user", "content": "There is a paper called an abundence of possums"},
    ],
    response_format={
            "type": "json_schema",
            "schema": ResearchPaperExtraction.model_json_schema(),
        },
    )

output = json.loads(completion.choices[0].message.content)
pretty_output = json.dumps(output, indent=2)


In [17]:
print(research_paper1)

{'title': 'An Abundance of Possums'}


In [19]:
print(type(output))
print(type(pretty_output))

<class 'dict'>
<class 'str'>


In [20]:
print(output)

{'title': 'An Abundance of Possums: The Biology and Conservation of Opossums in the Americas'}


In [5]:
import dspy
lm = dspy.LM("openai/gpt-4.1-nano-2025-04-14", api_key=api_key)
dspy.configure(lm=lm)

In [36]:
from typing import Literal

class Classify(dspy.Signature):
    """symptom discussion -> presence/absence of symptom talk"""

    excerpt: str = dspy.InputField()
    symptom: Literal["Positive", "Negative"] = dspy.OutputField()

classify = dspy.Predict(Classify)
classify(exerpt="This book was super fun to read, though not the last chapter.")

2025/07/31 00:25:01 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: []. Missing: ['excerpt'].


Prediction(
    symptom='Negative'
)

In [37]:
import pandas as pd
import dspy
df = pd.read_csv('data/status_data.csv').head(200)
devset = [dspy.Example(excerpt=row['text'], symptom=row['label']).with_inputs('excerpt') for _, row in df.iterrows()]
def metric(example, pred, trace=None):
    return pred.symptom == example.symptom
from dspy.evaluate import Evaluate
evaluate = Evaluate(devset=devset, metric=metric)
evaluate(classify)

2025/07/31 00:25:17 INFO dspy.evaluate.evaluate: Average Metric: 101 / 200 (50.5%)


50.5

In [38]:
# Import the optimizer
from dspy.teleprompt import MIPROv2

# Initialize optimizer
teleprompter = MIPROv2(
    metric=metric,
    auto="heavy", # Can choose between light, medium, and heavy optimization runs
)

# Optimize program
print(f"Optimizing program with MIPRO...")
optimized_program = teleprompter.compile(
    classify.deepcopy(),
    trainset=devset,
    max_bootstrapped_demos=2,
    max_labeled_demos=0,
    requires_permission_to_run=False,
)

# Save optimize program for future use
#optimized_program.save(f"mipro_optimized")

# Evaluate optimized program
print(f"Evaluate optimized program...")
evaluate(optimized_program, devset=devset[:])

2025/07/31 00:25:31 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING HEAVY AUTO RUN SETTINGS:
num_trials: 27
minibatch: True
num_fewshot_candidates: 18
num_instruct_candidates: 9
valset size: 160

2025/07/31 00:25:31 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/07/31 00:25:31 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/07/31 00:25:31 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=18 sets of demonstrations...


Optimizing program with MIPRO...
Bootstrapping set 1/18
Bootstrapping set 2/18


 10%|█         | 4/40 [00:00<00:00, 282.44it/s]


Bootstrapped 1 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
Bootstrapping set 3/18


 20%|██        | 8/40 [00:00<00:00, 1516.31it/s]


...

Average Metric: 101.00 / 160 (63.1%): 100%|██████████| 160/160 [00:27<00:00,  5.72it/s]

2025/07/31 00:29:44 INFO dspy.evaluate.evaluate: Average Metric: 101 / 160 (63.1%)
2025/07/31 00:29:44 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [52.5, 51.25, 64.38, 50.0, 59.38, 51.88, 63.12]
2025/07/31 00:29:44 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 64.38
2025/07/31 00:29:44 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2025/07/31 00:29:44 INFO dspy.teleprompt.mipro_optimizer_v2: 

2025/07/31 00:29:44 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 64.38!



Evaluate optimized program...


2025/07/31 00:29:52 INFO dspy.evaluate.evaluate: Average Metric: 124 / 200 (62.0%)


62.0

In [39]:
dspy.inspect_history(n=1)





[2025-07-31T00:29:52.383682]

System message:

Your input fields are:
1. `excerpt` (str):
Your output fields are:
1. `symptom` (Literal['Positive', 'Negative']):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## excerpt ## ]]
{excerpt}

[[ ## symptom ## ]]
{symptom}        # note: the value you produce must exactly match (no extra characters) one of: Positive; Negative

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given a detailed doctor-patient dialogue focused on symptom inquiry, generate a concise statement indicating whether the symptom discussed is present (positive) or absent (negative) based on the conversation. The output should clearly state the symptom status as either "Positive" or "Negative.


User message:

[[ ## excerpt ## ]]
P: Right now, with my mom drinking every now and then.

D: I see, okay. Um, alright, well that's all the questions I had, and uh, sorry one more. Have you b